# Multi-Layer Perceptron (MLP) on MNIST with PyTorch

A multi-layer perceptron (MLP) stacks linear layers separated by nonlinear activation functions,
letting it represent nonlinear decision boundaries that a single linear layer (logistic regression)
cannot. This notebook trains an MLP on MNIST digit classification and walks through the same core
pipeline pieces: data loading, model definition, loss functions, optimization via gradient descent,
and evaluation.

In [ ]:
%pip install Numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data: MNIST as a supervised learning problem

MNIST consists of 70,000 grayscale images (28x28 pixels) of handwritten digits 0-9, split into
60,000 training and 10,000 test examples. Formally, we treat this as a supervised learning problem:
we want to learn a function $f_\theta: \mathbb{R}^{784} \to \{0, \dots, 9\}$ that maps a flattened
pixel vector $x$ to a digit label $y$, by fitting parameters $\theta$ on labeled training pairs
$(x_i, y_i)$.

**Why flatten to 784 dimensions?** A plain logistic regression model has no notion of 2D spatial
structure — it only knows about linear combinations of input features. So we treat each of the
28x28=784 pixels as an independent feature and flatten the image into a vector. (A convolutional
network would instead exploit the 2D grid structure, but that's a different model class.)

**Why normalize pixel values?** Raw pixel intensities are integers in $[0, 255]$. We rescale them to
have roughly zero mean and unit variance using the dataset's known mean (0.1307) and standard
deviation (0.3081). Gradient-based optimizers converge faster and more stably when input features
are on a similar, well-centered scale — otherwise the loss surface becomes elongated/skewed in a
way that makes a single global learning rate work poorly across dimensions.

**Why mini-batches?** Computing gradients over all 60,000 examples at once (batch gradient descent)
is accurate but slow and memory-heavy. Using a single example at a time (pure SGD) is fast per step
but very noisy. Mini-batches (e.g. 64 examples) are the standard compromise: they give a
lower-variance, vectorized (fast on GPU/CPU via matrix ops) estimate of the true gradient while
still taking many update steps per epoch.

In [ ]:
# Compose the preprocessing pipeline: convert PIL image -> tensor, then normalize.
transform = transforms.Compose([
    transforms.ToTensor(),                       # scales pixels from [0, 255] ints to [0, 1] floats
    transforms.Normalize((0.1307,), (0.3081,)),   # standardize using MNIST's known mean/std
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Train examples: {len(train_dataset)}, Test examples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

# Visualize a few samples
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## 2. Model: a multi-layer perceptron

**Why stacking linear layers alone doesn't help.** If we stacked two linear layers with nothing in
between, $z = W_2(W_1 x + b_1) + b_2 = (W_2 W_1)x + (W_2 b_1 + b_2)$, which is *still* just an affine
function of $x$ — the composition collapses back into a single linear map, no more expressive than
logistic regression. The extra capacity only appears once we insert a **nonlinear activation
function** $\sigma$ between layers: $z = W_2\,\sigma(W_1 x + b_1) + b_2$. Now the network can
represent nonlinear decision boundaries, because the hidden layer first warps the input space before
the next linear layer draws its hyperplane.

**Activation function: ReLU.** We use $\text{ReLU}(z) = \max(0, z)$ in the hidden layers. Compared to
sigmoid/tanh, ReLU (a) is cheap to compute and differentiate, (b) has gradient exactly 1 for any
positive input, which avoids the *vanishing gradient* problem that sigmoid/tanh suffer from when
their input is far from 0 (their gradients shrink toward 0 in saturated regions, so backpropagated
gradients get multiplied by tiny numbers layer after layer and training stalls).

**Why this is more expressive, in principle.** The *universal approximation theorem* says a
feedforward network with even a single hidden layer of finite width and a non-polynomial activation
(like ReLU) can approximate any continuous function on a compact domain to arbitrary accuracy, given
enough hidden units. Logistic regression is the special case of zero hidden layers — restricted to
linear functions only. This is why MLPs can, in principle, fit patterns (like non-linearly separable
digit shapes) that logistic regression structurally cannot.

**Architecture.** We'll use two hidden layers: $784 \to 256 \to 128 \to 10$, with ReLU after each
hidden layer (but not after the final layer, since we still want raw logits going into
`CrossEntropyLoss`).

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=28 * 28, hidden_dims=(256, 128), num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], num_classes),  # raw logits, no activation here
        )

    def forward(self, x):
        x = self.flatten(x)  # (batch, 1, 28, 28) -> (batch, 784)
        return self.net(x)

model = MLP().to(device)
print(model)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

## 3. Loss function: cross-entropy as maximum likelihood

We want to find parameters $\theta$ that make the model assign high probability to the
correct label. The principled way to do this is **maximum likelihood estimation (MLE)**: choose
$\theta$ to maximize the probability the model assigns to the true labels across the training set,
assuming examples are i.i.d.:
$$\hat{\theta} = \arg\max_\theta \prod_{i=1}^{N} P_\theta(y_i \mid x_i)$$

Taking the log (monotonic, turns products into sums) and negating (to turn maximization into the
minimization that optimizers expect) gives the **negative log-likelihood**:
$$\mathcal{L}(\theta) = -\sum_{i=1}^{N} \log P_\theta(y_i \mid x_i)$$

For a single example with true class $y$ and predicted distribution $p = \text{softmax}(z)$, this is
exactly the **cross-entropy loss**:
$$\ell(z, y) = -\log p_y = -\log\left(\frac{e^{z_y}}{\sum_j e^{z_j}}\right) = -z_y + \log\sum_j e^{z_j}$$

This is also the cross-entropy $H(q, p) = -\sum_k q_k \log p_k$ between the true label's one-hot
distribution $q$ and the model's predicted distribution $p$ — cross-entropy is minimized (equals the
entropy of $q$, which is 0 since $q$ is one-hot) exactly when $p$ puts all its mass on the true
class. So minimizing cross-entropy loss over the dataset *is* maximum likelihood estimation for this
model.

**Why `nn.CrossEntropyLoss` and not manual softmax + `nn.NLLLoss`?** Computing $\log\sum_j e^{z_j}$
naively can overflow if logits are large. PyTorch's `CrossEntropyLoss` combines `log_softmax` and
`nll_loss` into one numerically stable operation (using the log-sum-exp trick internally), and it
expects raw logits — not probabilities — as input, which is why our model's `forward` returns `z`
directly.

## 4. Optimization: gradient descent

There's no closed-form solution for $\theta$ that minimizes cross-entropy loss (unlike, say,
ordinary least squares), so we minimize it iteratively with **gradient descent**: repeatedly step
the parameters in the direction that decreases the loss fastest, i.e. the negative gradient:
$$\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}(\theta)$$
where $\eta$ is the **learning rate**, controlling step size. Too large and training diverges/
oscillates; too small and training crawls.

Since $\mathcal{L}$ sums a loss term per example, its gradient is a sum of per-example gradients.
**Stochastic (mini-batch) gradient descent** approximates the full gradient using only the current
batch, which is unbiased in expectation and far cheaper to compute per step than using the whole
dataset.

We use **Adam** rather than plain SGD. Adam keeps running estimates of the first moment (mean) and
second moment (uncentered variance) of the gradient for each parameter and uses them to rescale each
parameter's effective learning rate individually — parameters with consistently large gradients get
smaller effective steps and vice versa. This adaptive per-parameter scaling generally makes Adam
converge faster and be less sensitive to the learning-rate choice than vanilla SGD, at the cost of
extra memory to store the moment estimates (negligible here).

Crucially, we never derive $\nabla_\theta \mathcal{L}$ by hand: PyTorch's **autograd** builds a
computation graph during the forward pass and computes exact gradients via reverse-mode automatic
differentiation (backpropagation) when we call `.backward()`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10

## 5. Training loop

One **epoch** is one full pass over the training set, processed batch by batch. For each batch, the
loop performs four steps that directly implement the theory above:

1. **`optimizer.zero_grad()`** — PyTorch accumulates gradients into `.grad` by default (useful for
   things like gradient accumulation across multiple batches), so we must explicitly clear them
   before each new batch, otherwise gradients from previous batches would leak into the current
   update.
2. **Forward pass** — `logits = model(x)` computes the model's output for the batch.
3. **`loss.backward()`** — autograd walks the computation graph backward from the scalar loss,
   applying the chain rule to compute the gradient of the loss with respect to every parameter,
   storing it in that parameter's `.grad`.
4. **`optimizer.step()`** — applies the Adam update rule using those gradients to move the
   parameters in the direction that reduces the loss.

We also put the model in **`.train()`** mode (a no-op for this MLP since it has no dropout/batchnorm layers, but it's good practice since it's required for models that behave differently at train vs. eval time).

In [ ]:
def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():  # no need to track gradients / build a graph for evaluation
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss_sum += criterion(logits, y).item() * x.size(0)
            preds = logits.argmax(dim=1)  # predicted class = highest-logit class (argmax of softmax too, since softmax is monotonic)
            correct += (preds == y).sum().item()
            total += x.size(0)
    return loss_sum / total, correct / total


train_losses, test_losses, test_accuracies = [], [], []

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()          # 1. clear stale gradients
        logits = model(x)              # 2. forward pass -> raw scores z
        loss = criterion(logits, y)    # cross-entropy loss for this batch
        loss.backward()                # 3. backward pass -> populate .grad via autograd
        optimizer.step()               # 4. Adam update: theta <- theta - adaptive_step * grad

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_dataset)
    test_loss, test_acc = evaluate(test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    print(f"Epoch {epoch:2d} | train loss {train_loss:.4f} | test loss {test_loss:.4f} | test acc {test_acc:.4f}")

## 6. Evaluation: the decision rule and why accuracy is a fair metric here

During evaluation we switch to **`model.eval()`** and wrap the loop in **`torch.no_grad()`**: we're
not updating parameters, so there's no need for autograd to record operations for backpropagation,
which saves memory and compute.

To turn the model's probability distribution into a single predicted digit, we take
$\hat{y} = \arg\max_k P(y=k \mid x) = \arg\max_k z_k$ (the argmax over probabilities and over raw
logits coincide because softmax is monotonic in each $z_k$). This is the **Bayes-optimal decision
rule under 0-1 loss**: if the model's probabilities were perfectly calibrated, always predicting the
most likely class minimizes the expected classification error rate.

Since MNIST's classes are (roughly) balanced — about 10% of examples per digit — plain **accuracy**
(fraction of correct predictions) is a reasonable summary metric here. (For imbalanced problems,
accuracy can be misleading and metrics like precision/recall/F1 per class are preferred; not an
issue for MNIST.)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(1, num_epochs + 1), train_losses, label="train loss")
axes[0].plot(range(1, num_epochs + 1), test_losses, label="test loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Loss curves")
axes[0].legend()

axes[1].plot(range(1, num_epochs + 1), test_accuracies, color="green")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("test accuracy")
axes[1].set_title("Test accuracy")

plt.tight_layout()
plt.show()

print(f"Final test accuracy: {test_accuracies[-1]:.4f}")

## 7. Sample predictions and takeaways

Finally, let's look at some individual predictions, including errors. Unlike logistic regression,
this MLP can represent nonlinear decision boundaries thanks to its hidden layers and ReLU
activations, which typically lets it reach around 97%-98% test accuracy on MNIST — a clear step up
from the ~92%-93% a purely linear model plateaus at, though still short of the >99% achievable with
convolutional networks that also exploit the image's 2D spatial structure.

In [ ]:
model.eval()
x_batch, y_batch = next(iter(test_loader))
with torch.no_grad():
    preds = model(x_batch.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_batch[i].squeeze(), cmap="gray")
    color = "green" if preds[i] == y_batch[i] else "red"
    ax.set_title(f"true {y_batch[i].item()} / pred {preds[i].item()}", color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()